# Hartree-Fock法で原子核を解いてみよう
ここではHartree-Fock(HF)法を用いて原子核の構造を計算します．
HFではハミルトニアンを
$$
H = E_{0} + \sum_{p'p} f_{p'p} \{a_{p'}^\dagger a_p\} + \frac{1}{4} \sum_{p'q'pq} \Gamma_{p'q'pq} \{a_{p'}^\dagger a_{q'}^\dagger a_q a_p\}
$$
とした時，$f_{p'p}$が対角となるように基底を選ぶと，それが単一スレーター行列の部分空間で$E_{0}$を最小化することと同じでした．
まずは計算に必要なライブラリをインポートします．

In [ ]:
from pathlib import Path
import os, sys

REPO_NAME = "Lecture_Hokkaido"
GITHUB_REPO = "https://github.com/Takayuki-Miyagi/Lecture_Hokkaido.git"
PACKAGE_DIR = "."

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        !git clone {GITHUB_REPO}
    %cd {REPO_NAME}
    %pip install -q -e .

def find_repo_root(package_dir=PACKAGE_DIR):
    cwd = Path.cwd().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "src" / package_dir).is_dir():
            return p

    raise RuntimeError(
        f"Could not find src/{package_dir}. "
        "Please run this notebook from inside the repository."
    )

repo_root = find_repo_root()
src_dir = repo_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print("repo root:", repo_root)
print("src added:", src_dir)


import constants
import LECs
from single_partile_orbit import Orbits
from two_body_space import TwoBodySpace
from model_space import ModelSpace
from partial_wave_basis import PartialWaveChannel, PartialWaveChannels
from ho_partial_wave_basis import HOPartialWaveChannel, HOPartialWaveChannels
from nn_pot import NNPotential
from ho_nn_pot import HONNPotential
from nuclear_operator import Operator
from hartree_fock import HartreeFock


まずは計算を行うためのパラメータを定義しましょう．
`emax`は球形な調和振動子基底における取りうる最大の量子数を指定しており，$\max(2n+l)$に対応します．
例えば，`emax`=0なら，中性子と陽子の$0s_{1/2}$のみを考えることになります．
`emax`=1なら，それらに加えて$0p_{3/2}$と$0p_{1/2}$も考慮することになります．
`emax`は計算結果がそれに依存しなくなるまで増やさなければなりません．

`Nmax`は2体相互作用の行列要素を計算する際に，その相対運動の量子数の最大値を指定するパラメータです．
2核子系のエネルギー量子数の和を越えることはないので，最大でも`2*emax`までの値を指定すれば十分です．

`Jrel_max`も2体相互作用の行列要素を計算する際に，その相対運動の角運動量の最大値を指定するパラメータです．
`Jrel_max`に依存しなくなるまで増やすことが理想的です．

`hw`は調和振動子のエネルギー$\hbar\omega$です．もし計算が収束していれば，`hw`にも依存しなくなるはずです．



In [ ]:
emax = 4
Nmax = 2*emax
Jrel_max = 4
hw = 20.0

まずは，運動量の部分波基底を定義して，利用するNN相互作用の行列要素を計算します．
ここは前回のハンズオンで行ったことと同じです．

In [ ]:
pw = PartialWaveChannels(Jmax=Jrel_max, NMesh=40, kmax=6)

parameters = LECs.N2LO_opt
filename = "N2LOopt.npz"
vpot_mom = NNPotential(pw)
if(os.path.exists(filename)):
    vpot_mom.load_npz_into(filename)
else:
    vpot_mom.set_potential(parameters)
    vpot_mom.save_npz(filename)
    

上ではNN相互作用の行列要素は
$$
\langle p' l' S J' Tz | V | p l S J Tz \rangle
$$
として与えられていました．
下では，それを調和振動子基底に変換しています．
つまり
$$
\langle p' l' S J' Tz | V | p l S J Tz \rangle \to \langle n' l' S J' Tz | V | n l S J Tz \rangle
$$
です．

In [ ]:
vpot_ho = HONNPotential(Nmax, Jrel_max, hw)
vpot_ho.set_nn_potential(vpot_mom)

下では，HFを解くための模型空間を定義しています．
`emax`, `hw`は上で定義したものと同じです．
ここではO16について解くので，reference stateとして$0s_{1/2}$, $0p_{3/2}$, $0p_{1/2}$を占有した状態を指定します．

In [ ]:
ms = ModelSpace()
ms.set_model_space_from_emax(emax)
ms.set_frequency(hw)
ms.set_reference_from_string("O16")
ms.assign_particle_hole()



次に空のOperatorクラスを作成します．
`tkin`は運動エネルギー，`vnn`はNN相互作用項，`ham`はトータルのハミルトニアンを格納するためのものです．
`tkin.set_kinetic_energy()`では運動エネルギーの行列要素を計算しています．ここは解析的に計算できる部分です．

次に先ほど計算しておいた相対座標HO基底でのNN相互作用の行列要素のTalmi-Moshinsky変換を行います．
ここでは
$$
\langle n' l' S J' Tz | V | n l S J Tz \rangle \to \langle p' q': J | V | pq :J \rangle
$$
をしています．

トータルのハミルトニアンは`ham`で与えられ，`ham` = `tkin` + `vnn`で計算されます．
今の段階ではハミルトニアンは
$$
H = \sum_{p'p} t_{p'p} a_{p'}^\dagger a_p + \frac{1}{4} \sum_{p'q'pq} V_{p'q'pq} a_{p'}^\dagger a_{q'}^\dagger a_q a_p
$$
で与えられています．

In [ ]:
tkin, vnn, ham = Operator(ms), Operator(ms), Operator(ms)
tkin.set_kinetic_term()
vnn.set_nn_interaction(vpot_ho)
ham = tkin + vnn

HFをするために，まずは`HartreeFock`クラスを作成します．
`HartreeFock`クラスの引数には，ハミルトニアンを与えます．
`hf.solve()`でHFを解きます．この時，内部では$f_{p'p}$のようなものを計算しているのですが，それはまだ`ham`に反映されていません．
この時に，`hf`の中にはHO -> HF変換行列が格納されます．

In [ ]:

hf = HartreeFock(ham)
hf.solve()

HFは解いたのですが，ハミルトニアンはまだHO基底で表現されています．
`hf.transform_operator_ho_to_hf(ham)`で，ハミルトニアンをHO基底からHF基底に変換します．
さらに，`ham.take_normal_ordering()`でハミルトニアンを正規順序に変換します．

In [ ]:
ham = hf.transform_operator_ho_to_hf(ham)
ham.take_normal_ordering()
print(f"HF: {ham.get_0bme():12.6f}")

`ham.take_normal_ordering()`を行った後，ハミルトニアンは
$$
H = E_{HF} + \sum_{p'p} f_{p'p} \{a_{p'}^\dagger a_p\} + \frac{1}{4} \sum_{p'q'pq} \Gamma_{p'q'pq} \{a_{p'}^\dagger a_{q'}^\dagger a_q a_p\}
$$
のように表現されます．ここで，$E_{HF}$はHF基底での0体行列要素，$f_{p'p}$はHF基底での1体行列要素，$\Gamma_{p'q'pq}$はHF基底での2体行列要素です．
さらに，平均場を超えた効果を取り入れるために，2次の摂動論的補正を計算します．
摂動補正は
$$
E^{(2)} = \frac{1}{4} \sum_{abij} \frac{|\Gamma_{abij}|^2}{f_{ii} + f_{jj} - f_{aa} - f_{bb}}
$$
で与えられます．ここで，$i,j$は占有軌道，$a,b$は非占有軌道です．
実際には$J$-coupled schemeで計算しているので，
$$
E^{(2)} = \sum_{a \le b} \sum_{i \le j}\sum_{J} \frac{(2J+1)|\Gamma_{abij}^{J}|^2}{f_{ii} + f_{jj} - f_{aa} - f_{bb}}
$$
で計算しています．

In [ ]:
e2 = hf.second_order_correction(ham)
print(f"2nd order correction: {e2:12.6f}, Total: {ham.get_0bme() + e2:12.6f}")

# Hands-on questions

***平均場のエネルギーと2次の摂動補正を比較してみましょう．***

***single-particle energyを見てみましょう．***

***`emax`や`hw`を変えて，HFやMBPT2を計算してplotしてみましょう．収束させることができるでしょうか？***

***他の原子核についても計算してみましょう．例えば，He4やCa40などはどうでしょうか？***